<a href="https://colab.research.google.com/github/isaacprueba/colab-notebook-automation-apps/blob/main/Scraping_Pages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install django


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 48.9 MB/s eta 0:00:00


In [10]:
# Instalación robusta para entornos Linux/Colab
print("Preparando entorno de navegación...")
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver
!pip install selenium
print("Dependencias listas.")

Preparando entorno de navegación...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
chromium-browser is already the newest version (1:85.0.418

In [13]:
import os
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from google.colab import drive
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

# 1. Configuración de Almacenamiento
def setup_drive_storage(folder_name="Scraper_Dynamic_System"):
    try:
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive', force_remount=True)
        base_path = f'/content/drive/My Drive/{folder_name}'
        if not os.path.exists(base_path):
            os.makedirs(base_path)
        print(f"[Drive] Almacenamiento listo en: {base_path}")
        return base_path
    except Exception as e:
        print(f"[!] Usando almacenamiento local: {e}")
        path = "/content/downloads"
        if not os.path.exists(path): os.makedirs(path)
        return path

# 2. Scraper Dinámico
class DynamicScraper:
    def __init__(self, base_url, storage_path, extensions):
        self.base_url = base_url
        self.domain = urlparse(base_url).netloc
        self.storage_path = storage_path
        self.extensions = extensions
        self.visited_urls = set()
        self.downloaded_files = set()

        options = Options()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--remote-debugging-port=9222')
        # Definimos explícitamente la ubicación del binario
        options.binary_location = "/usr/bin/chromium-browser"

        # Definimos el servicio con la ruta al driver instalado
        service = Service(executable_path="/usr/bin/chromedriver")

        try:
            self.driver = webdriver.Chrome(service=service, options=options)
        except Exception as e:
            print(f"[!] Error al iniciar navegador: {e}")
            raise

    def get_dynamic_soup(self, url):
        try:
            self.driver.get(url)
            time.sleep(5)
            return BeautifulSoup(self.driver.page_source, 'html.parser')
        except Exception: return None

    def download_resource(self, url):
        if url in self.downloaded_files: return
        try:
            res = requests.get(url, stream=True, timeout=10, verify=False)
            if res.status_code == 200:
                fname = os.path.basename(urlparse(url).path)
                if not fname: return
                fpath = os.path.join(self.storage_path, fname)
                if not os.path.exists(fpath):
                    with open(fpath, 'wb') as f:
                        for chunk in res.iter_content(8192): f.write(chunk)
                    print(f"[OK] {fname}")
                self.downloaded_files.add(url)
        except Exception: pass

    def process(self, url, depth=1):
        if depth < 0 or url in self.visited_urls: return
        self.visited_urls.add(url)
        print(f"[*] Rastreando: {url}")

        soup = self.get_dynamic_soup(url)
        if not soup: return

        for tag in soup.find_all(['a', 'img', 'video', 'source']):
            link = tag.get('href') or tag.get('src')
            if not link: continue

            full_url = urljoin(url, link)
            parsed = urlparse(full_url)
            ext = os.path.splitext(parsed.path)[1].lower().replace('.','')

            if ext in self.extensions:
                self.download_resource(full_url)
            elif parsed.netloc == self.domain and full_url not in self.visited_urls:
                self.process(full_url, depth - 1)

def run_scraper():
    print("--- MEGA SCRAPER DINÁMICO ---")
    url = input("URL (ej: https://site.com): ").strip()
    if not url.startswith('http'): url = 'https://' + url

    print("\nCategoría: 1.Docs 2.Imgs 3.Video 4.Todo")
    c = input("Selección: ")
    exts = {
        "1": ['pdf','docx','txt','xlsx'],
        "2": ['jpg','jpeg','png','webp','svg'],
        "3": ['mp4','avi','mov','webm'],
        "4": ['pdf','docx','txt','xlsx','jpg','jpeg','png','webp','svg','mp4','avi','mov','webm']
    }.get(c, ['pdf'])

    path = setup_drive_storage()
    scraper = DynamicScraper(url, path, exts)
    try:
        scraper.process(url, depth=1)
    finally:
        if hasattr(scraper, 'driver'):
            scraper.driver.quit()
        print("\n--- Tareas finalizadas ---")

if __name__ == "__main__":
    run_scraper()

--- MEGA SCRAPER DINÁMICO ---
URL (ej: https://site.com): informatica.umsa.bo

Categoría: 1.Docs 2.Imgs 3.Video 4.Todo
Selección: 1
[Drive] Almacenamiento listo en: /content/drive/My Drive/Scraper_Dynamic_System
[!] Error al iniciar navegador: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x5ab91ce232ca <unknown>
#1 0x5ab91c786b39 <unknown>
#2 0x5ab91c7c5b87 <unknown>
#3 0x5ab91c7c151e <unknown>
#4 0x5ab91c810d3e <unknown>
#5 0x5ab91c81042c <unknown>
#6 0x5ab91c7cfecb <unknown>
#7 0x5ab91c7d0cb1 <unknown>
#8 0x5ab91cde77f0 <unknown>
#9 0x5ab91cde5e6a <unknown>
#10 0x5ab91cdd0bb5 <unknown>
#11 0x5ab91cde6b2a <unknown>
#12 0x5ab91cdb9df0 <unknown>
#13 0x5ab91ce0e238 <unknown>
#14 0x5ab91ce0e3d5 <unknown>
#15 0x5ab91ce21e53 <unknown>
#16 0x77fd3979ba83 <unknown>



SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x5ab91ce232ca <unknown>
#1 0x5ab91c786b39 <unknown>
#2 0x5ab91c7c5b87 <unknown>
#3 0x5ab91c7c151e <unknown>
#4 0x5ab91c810d3e <unknown>
#5 0x5ab91c81042c <unknown>
#6 0x5ab91c7cfecb <unknown>
#7 0x5ab91c7d0cb1 <unknown>
#8 0x5ab91cde77f0 <unknown>
#9 0x5ab91cde5e6a <unknown>
#10 0x5ab91cdd0bb5 <unknown>
#11 0x5ab91cde6b2a <unknown>
#12 0x5ab91cdb9df0 <unknown>
#13 0x5ab91ce0e238 <unknown>
#14 0x5ab91ce0e3d5 <unknown>
#15 0x5ab91ce21e53 <unknown>
#16 0x77fd3979ba83 <unknown>
